# Trenowanie modelu

W tym notatniku w końcu wytrenujemy nasz model!

## Importy

Zacznijmy od zaimportowania potrzebnych bibliotek. Do trenowania modelu użyjemy PyTorch, jednej z najpopularniejszych bibliotek AI na świecie.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Urządzenie treningowe

W tej komórce sprawdzamy, czy ten komputer obsługuje sprzętowo przyspieszone trenowanie AI. Jeśli tak, użyjemy go, a jeśli nie, użyjemy CPU.

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f"Używam GPU")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print(f"Używam Apple NPU")
else:
  device = torch.device("cpu")
  print(f"Używam CPU")

## Wstępne przetwarzanie

Przed treningiem modelu musimy przygotować obrazy.

Robimy też coś sprytnego - ustawiamy, aby za każdym razem, gdy model widzi obraz, był on losowo obracany o 15 stopni. Dzięki temu model nie będzie próbował zapamiętywać dokładnych obrazów, tylko szukać bardziej ogólnych wzorców (tzn. nie zapamięta pudełka w jednej konkretnej orientacji, ale nauczy się, jak ogółem wyglądają pudełka). Ta technika nazywa się augmentacją danych.

In [ ]:
train_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),  # Upewnij się, że wszystkie obrazy mają ten sam rozmiar
    transforms.RandomRotation(degrees=15),  # Dodaj losowy obrót w ramach augmentacji danych
    transforms.ToTensor(),  # Konwertuj obrazy do formatu PyTorch
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Znormalizuj wartości pikseli do zakresu od -1 do 1
  ]
)

# Na teście chcemy użyć oryginalnych zdjęć bez augmentacji, więc tylko zmieniamy rozmiar i normalizujemy
test_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
  ]
)

## Podział train/test

Teraz dzielimy obrazy na zbiór treningowy i testowy. Zbioru treningowego użyjemy do trenowania modelu, a testowego do sprawdzenia, jak dobrze działa.

In [ ]:
# Zbiór danych z przetwarzaniem treningowym
full_dataset_for_train = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=train_transforms
)

# Zbiór danych z przetwarzaniem testowym
full_dataset_for_test = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=test_transforms
)

# Chcemy podzielić zbiór tak, aby proporcje zdjęć forward/obstacle były takie same w train i test

idxs = np.arange(len(full_dataset_for_train))  # Pobierz indeksy wszystkich próbek w zbiorze
labels = full_dataset_for_train.targets  # Pobierz etykiety każdej próbki (0 dla forward, 1 dla obstacle)

train_idx, test_idx = train_test_split(
  idxs,
  test_size=0.2,  # Użyj 20% danych do testowania
  stratify=labels  # Upewnij się, że rozkład klas jest taki sam w obu zbiorach
)

# Utwórz zbiory train i test na podstawie odpowiednich indeksów
train_dataset = torch.utils.data.Subset(full_dataset_for_train, train_idx)
test_dataset = torch.utils.data.Subset(full_dataset_for_test, test_idx)

# Utwórz data loadery dla zbiorów train i test
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
print(f"Zdjęcia forward: {np.sum(np.array(labels)[train_idx] == 0)}")
print(f"Zdjęcia obstacle: {np.sum(np.array(labels)[train_idx] == 1)}")

In [ ]:
print(f"Procent forward: {np.sum(np.array(labels) == 0) / len(labels) * 100:.2f}%")
print(f"Procent obstacle: {np.sum(np.array(labels) == 1) / len(labels) * 100:.2f}%")

## Architektura modelu

Teraz definiujemy architekturę naszego modelu!

Używamy głównie trzech typów warstw:
- `Conv2d` - stosuje konwolucję wyjaśnioną podczas wykładu
- `MaxPool2d` - zastępuje grupy pikseli ich wartościami maksymalnymi, przez co obraz jest mniejszy
- `BatchNorm2d` - robi trochę sprytnej matematyki, dzięki której trening jest bardziej stabilny ;)
- `ReLU` - stosuje funkcję ReLU(val) = max(0, val)

Na końcu używamy też:
- `AdaptiveAvgPool2d` - bierze średnią z całych warstw obrazu (kanałów)
- `Flatten` - przygotowuje dane dla warstwy `Linear` (zamienia dane na tablicę 1D)
- `Linear` - zwykła, w pełni połączona warstwa sieci neuronowej
- `Dropout` - losowo wyłącza część neuronów podczas treningu, więc sieć musi nauczyć się działać bez nich (zwiększa stabilność, zmniejsza przeuczenie)

In [ ]:
model = nn.Sequential(
  nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, padding="same"),
  nn.BatchNorm2d(num_features=64),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.AdaptiveAvgPool2d(output_size=1),
  nn.Flatten(),
  nn.Linear(in_features=256, out_features=128), nn.ReLU(),
  nn.Dropout(p=0.5),
  nn.Linear(in_features=128, out_features=64), nn.ReLU(),
  nn.Dropout(p=0.5),
  nn.Linear(in_features=64, out_features=1)
).to(device)

## Parametry uczenia

In [ ]:
# Ta funkcja mówi modelowi, jak mierzyć jego błędy (loss). Jeśli |(prawdziwa etykieta) - (predykcja modelu)| jest duża, loss będzie duży
criterion = nn.BCEWithLogitsLoss()

# Ta funkcja mówi modelowi, jak zmieniać się na podstawie popełnionych błędów. Będzie próbował aktualizować się tak, aby zmniejszać loss
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Ile razy model przejdzie przez cały zbiór treningowy, próbując się poprawiać
n_epochs = 50

In [ ]:
best_eval_loss = float("inf")

for epoch in range(n_epochs):
  running_loss = 0.0

  model.train()  # Niektóre warstwy zachowują się inaczej podczas treningu i ewaluacji, więc musimy ustawić tryb treningowy

  for i, data in enumerate(train_loader):
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

    # Mówimy modelowi, aby zapomniał poprzednie błędy (gradienty) i zaczął liczyć je od nowa
    optimizer.zero_grad()

    # Pobieramy predykcje modelu dla bieżących obrazów i porównujemy je z etykietami, aby obliczyć loss
    outputs = model(inputs)
    loss = criterion(outputs, labels)

    # Mówimy modelowi, aby zaktualizował się na podstawie uzyskanego loss i próbował go zmniejszyć
    loss.backward()
    optimizer.step()

    # Śledzimy loss, aby widzieć postęp modelu
    running_loss += loss.item()
  
  model.eval()  # Teraz ustawiamy tryb ewaluacji, żeby niektóre warstwy działały poprawnie

  # Sprawdzamy, jak dobrze model radzi sobie na obrazach, na których go nie trenujemy
  with torch.no_grad():
    eval_loss = 0.0

    for i, data in enumerate(test_loader):
      inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

      outputs = model(inputs)
      loss = criterion(outputs, labels)
      eval_loss += loss.item()
  
  print(f"Epoka {epoch+1}, Loss: {running_loss/len(train_loader)}, Eval Loss: {eval_loss/len(test_loader)}")

  # Zapisujemy najlepszy model do tej pory
  if eval_loss < best_eval_loss:
    best_eval_loss = eval_loss
    torch.save(model.state_dict(), "best_model.pth")

## Ewaluacja

In [ ]:
all_predictions = []
all_labels = []

# Przywróć najlepszy model
model.load_state_dict(torch.load("best_model.pth"))

model.eval()

# Sprawdź, jak dobrze model działa na zbiorze testowym
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
    outputs = model(inputs)
    predictions = torch.sigmoid(outputs) > 0.5
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

all_predictions = np.array(all_predictions).flatten()
all_labels = np.array(all_labels).flatten()

# Oblicz metryki ewaluacji
accuracy = np.sum(all_predictions == all_labels.astype(bool)) / len(all_labels)
precision = np.sum((all_predictions == 1) & (all_labels == 1)) / np.sum(all_predictions == 1)
recall = np.sum((all_predictions == 1) & (all_labels == 1)) / np.sum(all_labels == 1)

print(f"Dokładność (accuracy): {accuracy:.4f}")
print(f"Precyzja (precision): {precision:.4f}")
print(f"Czułość (recall): {recall:.4f}")

Znaczenie metryk:
- accuracy - jak często model miał rację
- precision - spośród wszystkich obrazów sklasyfikowanych przez model jako przeszkody, ile faktycznie było przeszkodami
- recall - spośród wszystkich obrazów, które naprawdę były przeszkodami, ile model wykrył

Jeśli precision jest niskie, robot będzie często skręcał nawet wtedy, gdy droga jest wolna.
Jeśli recall jest niskie, robot będzie wpadał na przeszkody.

**Model jest teraz gotowy do testów na robocie!**